# Direct fine-tuning: MoViNet-A0 for binary shoplifting detection

This Colab-ready notebook downloads a balanced Shoplifting/Normal subset from the Hugging Face UCF-Crime mirror, fine-tunes a small MoViNet-A0 student, evaluates it, visualizes results, and benchmarks inference. Video decoding is CPU-side; model training/inference runs on CUDA.

In [ ]:
# Shoplifting detection — direct fine-tuning
# Colab: Runtime → Change runtime type → GPU
!nvidia-smi -L
!pip -q install -U "transformers>=4.45" "accelerate>=0.34" "huggingface_hub>=0.25" "decord>=0.6.0" "scikit-learn>=1.4" "pandas>=2.0" "matplotlib>=3.8" "seaborn>=0.13" "tqdm>=4.66"

In [ ]:
import os, json, random, time, math, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, roc_auc_score, average_precision_score, confusion_matrix, classification_report, roc_curve, precision_recall_curve
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm
import decord
decord.bridge.set_bridge("native")
assert torch.cuda.is_available(), "Enable a Colab GPU."
DEVICE=torch.device("cuda")
torch.backends.cuda.matmul.allow_tf32=True
torch.backends.cudnn.benchmark=True
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
ROOT=Path("/content/shoplifting_project"); DATA=ROOT/"data"; CKPT=ROOT/"checkpoints"; FIG=ROOT/"figures"
for p in (DATA,CKPT,FIG): p.mkdir(parents=True,exist_ok=True)
UCF_REPO="jinmang2/ucf_crime"
TRAIN_LIST="UCF_Crimes-Train-Test-Split/Action_Recognition_splits/train_001.txt"
VAL_LIST="UCF_Crimes-Train-Test-Split/Action_Recognition_splits/test_001.txt"
STUDENT_ID="kfkas/movinet-a0-stream-pytorch"
LABELS=["normal","shoplifting"]; LABEL2ID={"normal":0,"shoplifting":1}; ID2LABEL={0:"normal",1:"shoplifting"}
FRAMES=16; IMG_SIZE=172; EPOCHS=5; LR=2e-4; WEIGHT_DECAY=1e-4
VRAM=torch.cuda.get_device_properties(0).total_memory/2**30
BATCH_SIZE=8 if VRAM>=20 else 4 if VRAM>=10 else 2
NUM_WORKERS=2
print(torch.cuda.get_device_name(0), f"{VRAM:.1f} GB VRAM; batch={BATCH_SIZE}")

In [ ]:
def list_items(path):
    local=hf_hub_download(repo_id=UCF_REPO,filename=path,repo_type="dataset")
    return [s.strip() for s in Path(local).read_text(errors="ignore").splitlines() if s.strip().endswith(".mp4")]
def select_split(list_path):
    items=list_items(list_path); pos=[]; neg=[]
    for rel in items:
        if rel.startswith("Shoplifting/"): pos.append((rel,1))
        elif "Normal_Videos" in rel: neg.append((rel,0))
    random.shuffle(pos); random.shuffle(neg)
    n=min(len(pos),len(neg))
    return pos[:n]+neg[:n]
train_items, val_items=select_split(TRAIN_LIST), select_split(VAL_LIST)
print("train:",len(train_items),"val:",len(val_items))
def ensure_download(items, split):
    out=[]
    for rel,y in tqdm(items,desc=f"download {split}"):
        p=hf_hub_download(repo_id=UCF_REPO,filename=rel,repo_type="dataset",local_dir=str(DATA/split))
        out.append((str(p),y,rel))
    return out
train_files=ensure_download(train_items,"train"); val_files=ensure_download(val_items,"val")
print("downloaded",len(train_files),len(val_files))

In [ ]:
def read_clip(path,n=FRAMES):
    vr=decord.VideoReader(path,ctx=decord.cpu(0))
    idx=np.linspace(0,len(vr)-1,n).astype(np.int64)
    return vr.get_batch(idx).asnumpy()
def prep(frames, size=IMG_SIZE, flip=False):
    x=torch.from_numpy(frames).permute(0,3,1,2).float()/255
    x=F.interpolate(x,size=(size,size),mode="bilinear",align_corners=False)
    if flip and random.random()<0.5: x=torch.flip(x,[-1])
    return x
class VideoDS(Dataset):
    def __init__(self,items,train=False): self.items=items; self.train=train
    def __len__(self): return len(self.items)
    def __getitem__(self,i):
        path,y,rel=self.items[i]
        try: x=prep(read_clip(path),flip=self.train)
        except Exception:
            x=torch.zeros(FRAMES,3,IMG_SIZE,IMG_SIZE); y=0
        return x,torch.tensor(y),rel
train_loader=DataLoader(VideoDS(train_files,True),batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=True)
val_loader=DataLoader(VideoDS(val_files),batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=True)
print("batches",len(train_loader),len(val_loader))

In [ ]:
from transformers import AutoModelForVideoClassification
model=AutoModelForVideoClassification.from_pretrained(STUDENT_ID,trust_remote_code=True,num_labels=2,ignore_mismatched_sizes=True,label2id=LABEL2ID,id2label=ID2LABEL).to(DEVICE)
print("parameters:",sum(p.numel() for p in model.parameters())/1e6,"M")

### Training
Best checkpoint is selected by validation F1. The deployment model is only the small MoViNet student.

In [ ]:
def logits(out): return out.logits if hasattr(out,"logits") else out
scaler=torch.amp.GradScaler("cuda")
opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS)
history=[]
best=-1
def evaluate(m,loader):
    m.eval(); ys=[]; ps=[]; probs=[]; total=0
    with torch.no_grad():
        for x,y,_ in loader:
            x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE)
            with torch.autocast("cuda",dtype=torch.float16):
                o=logits(m(pixel_values=x))
            p=o.softmax(-1)[:,1]
            ys+=y.cpu().tolist(); probs+=p.float().cpu().tolist(); ps+=(o.argmax(-1).cpu().tolist()); total+=len(y)
    acc=accuracy_score(ys,ps); bal=balanced_accuracy_score(ys,ps)
    pr,re,f1,_=precision_recall_fscore_support(ys,ps,average="binary",zero_division=0)
    auc=roc_auc_score(ys,probs) if len(set(ys))>1 else float("nan")
    ap=average_precision_score(ys,probs) if len(set(ys))>1 else float("nan")
    return dict(accuracy=acc,balanced_accuracy=bal,precision=pr,recall=re,f1=f1,roc_auc=auc,average_precision=ap,y=ys,p=ps,prob=probs)
for epoch in range(1,EPOCHS+1):
    model.train(); run=0
    for x,y,_ in tqdm(train_loader,desc=f"epoch {epoch}"):
        x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE); opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda",dtype=torch.float16): loss=F.cross_entropy(logits(model(pixel_values=x)),y)
        scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update(); run+=loss.item()
    sched.step(); ev=evaluate(model,val_loader); ev["loss"]=run/len(train_loader); ev["epoch"]=epoch; history.append(ev)
    print({k:round(v,4) for k,v in ev.items() if isinstance(v,(int,float))})
    if ev["f1"]>best:
        best=ev["f1"]; torch.save({"state_dict":model.state_dict(),"history":history},CKPT/"movinet_direct_best.pt")
pd.DataFrame([{k:v for k,v in h.items() if k not in ("y","p","prob")} for h in history]).to_csv(ROOT/"direct_history.csv",index=False)

In [ ]:
h=pd.DataFrame([{k:v for k,v in z.items() if k not in ("y","p","prob")} for z in history])
fig,ax=plt.subplots(1,2,figsize=(14,4)); ax[0].plot(h.epoch,h.loss,marker="o",label="train loss"); ax[0].set_title("Training loss"); ax[0].grid(alpha=.2)
ax[1].plot(h.epoch,h.f1,marker="o",label="val F1"); ax[1].plot(h.epoch,h.balanced_accuracy,marker="o",label="val balanced acc"); ax[1].legend(); ax[1].set_title("Validation metrics"); ax[1].grid(alpha=.2); plt.show()
ev=evaluate(model,val_loader); cm=confusion_matrix(ev["y"],ev["p"])
plt.figure(figsize=(5,4)); sns.heatmap(cm,annot=True,fmt="d",xticklabels=LABELS,yticklabels=LABELS); plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix"); plt.show()
fpr,tpr,_=roc_curve(ev["y"],ev["prob"]); prec,rec,_=precision_recall_curve(ev["y"],ev["prob"])
fig,ax=plt.subplots(1,2,figsize=(12,4)); ax[0].plot(fpr,tpr); ax[0].plot([0,1],[0,1],"--"); ax[0].set_title(f"ROC AUC={ev['roc_auc']:.3f}"); ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR")
ax[1].plot(rec,prec); ax[1].set_title(f"PR AP={ev['average_precision']:.3f}"); ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision"); plt.show()
print(classification_report(ev["y"],ev["p"],target_names=LABELS,digits=4))

In [ ]:
path,y,rel=random.choice(val_files); frames=read_clip(path,8)
fig,axs=plt.subplots(2,4,figsize=(14,7))
for ax,fr in zip(axs.ravel(),frames): ax.imshow(fr); ax.axis("off")
fig.suptitle(f"sample: {LABELS[y]} — {rel}",fontsize=14); plt.show()

In [ ]:
model.eval(); xb=next(iter(val_loader))[0].to(DEVICE)
for _ in range(5):
    with torch.no_grad(),torch.autocast("cuda",dtype=torch.float16): _=logits(model(pixel_values=xb))
torch.cuda.synchronize(); t0=time.perf_counter()
N=30
with torch.no_grad():
    for _ in range(N):
        with torch.autocast("cuda",dtype=torch.float16): _=logits(model(pixel_values=xb))
torch.cuda.synchronize(); elapsed=time.perf_counter()-t0
print(f"batch latency: {elapsed/N*1000:.2f} ms | per-clip: {elapsed/N/len(xb)*1000:.2f} ms | throughput: {N*len(xb)/elapsed:.2f} clips/s")

### Notes
- Dataset source: `jinmang2/ucf_crime` (UCF-Crime mirror). The notebook downloads only the Shoplifting and Normal paths listed in the selected split files.
- For a smoke test, set `EPOCHS=1` and slice `train_files=train_files[:20]`, `val_files=val_files[:10]` before constructing loaders.
- Check the dataset's terms before redistribution or commercial use.